# Exploring Ethereum mainnet on BigQuery

Dataset: `bigquery-public-data.goog_blockchain_ethereum_mainnet_us`

**The cost model in three sentences.** BigQuery charges by bytes *scanned*
(first 1 TB per month is free), and `LIMIT` does **not** reduce scanning.
Cost is controlled by selecting few columns and filtering on the partition
column `block_timestamp` (all tables here are partitioned by month on it).
`dry_run()` shows what a query *would* scan for free, and `run_query()`
refuses anything that would bill more than `max_gb` (default 10 GB).

In [1]:
import pandas as pd

from eth_graph_research.bq import DATASET, dry_run, run_query

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
DATASET

'bigquery-public-data.goog_blockchain_ethereum_mainnet_us'

## What tables exist?

`INFORMATION_SCHEMA` queries are metadata-only and cost next to nothing.

In [2]:
run_query(f'''
SELECT table_name
FROM `{DATASET}.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
''')

Scanned 0.010 GB (billed 0.010 GB)


,table_name
0,accounts
1,accounts_state
2,accounts_state_by_address
3,blocks
4,decoded_events
5,logs
6,receipts
7,token_transfers
8,traces
9,transactions


## What columns do the key tables have?

Keep this DataFrame handy — it is the ground truth for column names and
types when you write your own queries.

In [3]:
schema = run_query(f'''
SELECT table_name, column_name, data_type
FROM `{DATASET}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name IN ('blocks', 'transactions', 'token_transfers')
ORDER BY table_name, ordinal_position
''')
schema

Scanned 0.010 GB (billed 0.010 GB)


,table_name,column_name,data_type
0,blocks,block_hash,STRING
1,blocks,block_number,INT64
2,blocks,block_timestamp,TIMESTAMP
3,blocks,parent_hash,STRING
4,blocks,size,INT64
5,blocks,extra_data,STRING
6,blocks,gas_limit,INT64
7,blocks,gas_used,INT64
8,blocks,base_fee_per_gas,INT64
9,blocks,mix_hash,STRING


## Dry runs: check cost before you spend it

Three estimates for the `transactions` table:
1. `SELECT *`, no filter — scans the entire table (terabytes!).
2. Same with `LIMIT 10` — **identical cost**. LIMIT is applied after scanning.
3. Two columns + one month of `block_timestamp` — a tiny fraction.

Rule of thumb: never run a query against a big table without a
`block_timestamp` filter, and always `dry_run` anything new.

In [4]:
dry_run(f'SELECT * FROM `{DATASET}.transactions`')
dry_run(f'SELECT * FROM `{DATASET}.transactions` LIMIT 10')
dry_run(f'''
SELECT transaction_hash, from_address
FROM `{DATASET}.transactions`
WHERE block_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
''')

Dry run: would scan 4,229.857 GB
Dry run: would scan 4,229.857 GB
Dry run: would scan 8.543 GB


8.5430232

## Example 1 — blocks from the last day

`blocks` is a small table; a day of it is cheap. Partitions are monthly, so
a 1-day filter still scans the whole current month's partition for the
selected columns — that's fine here.

In [5]:
run_query(f'''
SELECT
  COUNT(*)              AS n_blocks,
  MIN(block_number)     AS first_block,
  MAX(block_number)     AS last_block,
  ROUND(AVG(gas_used))  AS avg_gas_used
FROM `{DATASET}.blocks`
WHERE block_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
''')

Scanned 0.001 GB (billed 0.010 GB)


,n_blocks,first_block,last_block,avg_gas_used
0,7100,25634889,25641988,30371555.0


## Example 2 — transactions per day, last 7 days

The same shape as Google's own example query for this dataset.

In [6]:
run_query(f'''
SELECT
  TIMESTAMP_TRUNC(block_timestamp, DAY) AS day,
  COUNT(*) AS txn_count
FROM `{DATASET}.transactions`
WHERE block_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY day
ORDER BY day
''')

Scanned 0.110 GB (billed 0.110 GB)


,day,txn_count
0,2026-07-23 00:00:00+00:00,2319537
1,2026-07-24 00:00:00+00:00,2552428
2,2026-07-25 00:00:00+00:00,1998222
3,2026-07-26 00:00:00+00:00,1538710
4,2026-07-27 00:00:00+00:00,1871418
5,2026-07-28 00:00:00+00:00,1664214
6,2026-07-29 00:00:00+00:00,1734060
7,2026-07-30 00:00:00+00:00,10195


## Example 3 — a peek at recent individual transactions

Selecting few columns keeps the scan small; `value` is in wei
(1 ETH = 1e18 wei). The LIMIT here is for display only — the
`block_timestamp` filter is what keeps it cheap.

In [7]:
run_query(f'''
SELECT
  block_timestamp,
  transaction_hash,
  from_address,
  to_address,
  SAFE_CAST(value AS FLOAT64) / 1e18 AS value_eth
FROM `{DATASET}.transactions`
WHERE block_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)
ORDER BY block_timestamp DESC
LIMIT 20
''')

Scanned 0.010 GB (billed 0.010 GB)


,block_timestamp,transaction_hash,from_address,to_address,value_eth
0,2026-07-30 00:09:59+00:00,0x03074a2f8e60d14f75fe7970682a9aecfd6d35090025...,0x8e6b3c9c72ca592bbab470f761fa869c174e9ea0,0x4313c378cc91ea583c91387b9216e2c03096b27f,0.100000
1,2026-07-30 00:09:59+00:00,0x8434a92f9b62e5dd5d3d94b106c0f0afbd776c79ce3d...,0x8e6b3c9c72ca592bbab470f761fa869c174e9ea0,0x4313c378cc91ea583c91387b9216e2c03096b27f,0.100000
2,2026-07-30 00:09:59+00:00,0xe69ec3dd67d1d9f35d8d4ad379653797b9ecbea88381...,0x5050f69a9786f081509234f1a7f4684b5e5b76c9,0xff00000000000000000000000000000000008453,0.000000
3,2026-07-30 00:09:59+00:00,0x7ef2c24294cc51881f28253d9824fc0713e40efd9efb...,0x315d2ee4fccda0def532ef4108ff57204f8d9eba,0x447a03c131c0a97a8b8d548e3cd81aec4ce05d73,0.000000
4,2026-07-30 00:09:59+00:00,0x40dd8d27bca9615838cf65c681a67e2a48bc0a5a9036...,0x12ef5e054878b225fe3d36e5aa96a3ba41eac385,0xe6026f60efc3d0e714f1b89c596aa484ffbd25c8,53.627930
5,2026-07-30 00:09:59+00:00,0x4402a4b3fe6d39c760ebaf8e000ed444b51adf757b2d...,0x56e7d6c041b74c1fd2c42380dbffbd9c7df21c1d,0x150a3ba63ecad7ad8105140a87a2849eae1baefa,0.000000
6,2026-07-30 00:09:59+00:00,0x8dfb5edd15ee78a86148d16257decdf3f872f9410cf8...,0x54c2f9da85478ba2a13d0409fb938ae4ce24d96f,0xdac17f958d2ee523a2206206994597c13d831ec7,0.000000
7,2026-07-30 00:09:59+00:00,0xc2178997a3d69be91d6fd2666b438a03e5df8a1feff7...,0x18e296053cbdf986196903e889b7dca7a73882f6,0xdac17f958d2ee523a2206206994597c13d831ec7,0.000000
8,2026-07-30 00:09:59+00:00,0x283f24311bf6addf7164813f56f039ee813936e7a84f...,0x4e9141d2fb79b2a94a0256283f1547c7d6a12e7f,0x447a03c131c0a97a8b8d548e3cd81aec4ce05d73,0.000000
9,2026-07-30 00:09:59+00:00,0x36cfcfec31f8aec32575e981dda5c6371b5f83de5d9b...,0x58f25fa92926a512848567848f7624f449939cff,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,0.000000


## Example 4 — busiest token contracts in the last day

`token_transfers` is decoded ERC-20 `Transfer` events. `address` is the
token contract emitting the event. (If the schema cell above shows a
different column name, use that.)

In [8]:
run_query(f'''
SELECT
  address AS token_contract,
  COUNT(*) AS transfers
FROM `{DATASET}.token_transfers`
WHERE block_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
GROUP BY token_contract
ORDER BY transfers DESC
LIMIT 10
''')

Scanned 0.185 GB (billed 0.186 GB)


,token_contract,transfers
0,0xdac17f958d2ee523a2206206994597c13d831ec7,984861
1,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,699362
2,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,263426
3,0xc80b4457fccd49e7c3a2f5566dd1b827d85f6c98,165137
4,0xbeef007ecfbfdf9b919d0050821a9b6dbd634ff0,87298
5,0x647a139b234dcf9f91b1b749993604e715d3acb8,67757
6,0xaa8a56638b9f91fffa3188693731a8fcbcf40a3b,54618
7,0x2831b7136c49c23f53e3da38693fe00af1481b2a,40188
8,0x1aed8c8e8f5ac86800b1e914d797929f0f93f9c1,35031
9,0x56dff0942be7e6f3300279301776bb25ec7858fc,31336


## Where next

- Cross-reference token contracts against `crypto_ethereum.tokens`
  (the community dataset) or `decoded_events` to get symbols/decimals.
- For graph research: `transactions` (`from_address` → `to_address`) and
  `token_transfers` are the edge lists. Decide on a time window, dry-run
  the extraction query, then export.
- Raise `max_gb` per call only when a dry run justifies it:
  `run_query(sql, max_gb=50)`.